In [1]:
# ==========================================================
# Agent-Based Simulation of Long-Term Success
# Cell 1 - Import Libraries
# ==========================================================

import random
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mesa import Agent, Model
from mesa.datacollection import DataCollector

# Reproducibility
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ==========================================================
# Cell 2 - Global Simulation Parameters
# ==========================================================

# ----------------------------------------------------------
# Population
# ----------------------------------------------------------

TRAIT_LEVELS = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

TRAIT_NAMES = [
    "Cognitive Ability",
    "Conscientiousness",
    "Curiosity",
    "Risk Tolerance"
]

PERSONALITY_COMBINATIONS = list(
    itertools.product(
        TRAIT_LEVELS.keys(),
        repeat=len(TRAIT_NAMES)
    )
)

NUM_PERSONALITY_TYPES = len(PERSONALITY_COMBINATIONS)

AGENTS_PER_PERSONALITY = 10

NUM_AGENTS = NUM_PERSONALITY_TYPES * AGENTS_PER_PERSONALITY

assert NUM_PERSONALITY_TYPES == 81
assert NUM_AGENTS == 810

# ----------------------------------------------------------
# Simulation Length
# ----------------------------------------------------------

SIMULATION_DAYS = 3650          # 10 years

# ----------------------------------------------------------
# Initial Life Capitals
# ----------------------------------------------------------

INITIAL_MONEY = 100.0
INITIAL_KNOWLEDGE = 50.0
INITIAL_HEALTH = 100.0
INITIAL_NETWORK = 10.0

# ----------------------------------------------------------
# Living Cost
# ----------------------------------------------------------

LIVING_COST = 2.0

# Debt is allowed.
ALLOW_DEBT = True

# No interest in baseline experiment.
DEBT_INTEREST = 0.0

# ----------------------------------------------------------
# Personality Multipliers
# ----------------------------------------------------------
#
# Every personality trait uses the same
# three-level scale throughout the project.
#
# Low
# Medium
# High
#
# These multipliers will later affect
# learning, working, networking,
# entrepreneurship, etc.
# ----------------------------------------------------------

PERSONALITY_MULTIPLIER = {
    "Low": 0.8,
    "Medium": 1.0,
    "High": 1.2
}

# ----------------------------------------------------------
# Environment
# ----------------------------------------------------------

EDUCATION_QUALITY = 1.0
ECONOMIC_GROWTH = 1.0
HEALTHCARE_QUALITY = 1.0
SOCIAL_MOBILITY = 1.0

RANDOM_EVENT_PROBABILITY = 0.02

# ----------------------------------------------------------
# Action Names
# ----------------------------------------------------------

ACTIONS = [
    "Work",
    "Study",
    "Exercise",
    "Network",
    "Rest",
    "Entrepreneurship"
]

print("Simulation parameters initialized.")
print(f"Personality Types : {NUM_PERSONALITY_TYPES}")
print(f"Agents            : {NUM_AGENTS}")
print(f"Simulation Days   : {SIMULATION_DAYS}")

Simulation parameters initialized.
Personality Types : 81
Agents            : 810
Simulation Days   : 3650


In [3]:
# ==========================================================
# Cell 3 - Generate the 81 Personality Combinations
# ==========================================================

# Generate every possible personality combination.
# There are 3^4 = 81 combinations.

personality_profiles = []

for profile_id, combination in enumerate(PERSONALITY_COMBINATIONS, start=1):

    profile = {
        "Profile_ID": profile_id,
        "Cognitive Ability": combination[0],
        "Conscientiousness": combination[1],
        "Curiosity": combination[2],
        "Risk Tolerance": combination[3]
    }

    personality_profiles.append(profile)

# Convert to DataFrame for easy viewing
personality_df = pd.DataFrame(personality_profiles)

print(f"Total personality profiles: {len(personality_df)}")

assert len(personality_df) == 81

display(personality_df.head(10))

Total personality profiles: 81


,Profile_ID,Cognitive Ability,Conscientiousness,Curiosity,Risk Tolerance
0,1,Low,Low,Low,Low
1,2,Low,Low,Low,Medium
2,3,Low,Low,Low,High
3,4,Low,Low,Medium,Low
4,5,Low,Low,Medium,Medium
5,6,Low,Low,Medium,High
6,7,Low,Low,High,Low
7,8,Low,Low,High,Medium
8,9,Low,Low,High,High
9,10,Low,Medium,Low,Low


In [4]:
# ==========================================================
# Cell 4 - Agent Class
# ==========================================================

class Person(Agent):

    def __init__(self, model, unique_id, personality_profile):

        super().__init__(model)

        # ----------------------------
        # Identity
        # ----------------------------

        self.agent_id = unique_id
        self.profile_id = personality_profile["Profile_ID"]

        # ----------------------------
        # Fixed Personality Traits
        # ----------------------------

        self.cognitive = personality_profile["Cognitive Ability"]

        self.conscientiousness = personality_profile["Conscientiousness"]

        self.curiosity = personality_profile["Curiosity"]

        self.risk_tolerance = personality_profile["Risk Tolerance"]

        # ----------------------------
        # Dynamic Life Capitals
        # ----------------------------

        self.money = INITIAL_MONEY

        self.knowledge = INITIAL_KNOWLEDGE

        self.health = INITIAL_HEALTH

        self.network = INITIAL_NETWORK

        # ----------------------------
        # Other State
        # ----------------------------

        self.age = 0

        self.current_action = None

    def __repr__(self):

        return (
            f"Agent {self.agent_id} "
            f"(Profile {self.profile_id}) | "
            f"Money={self.money:.1f}, "
            f"Knowledge={self.knowledge:.1f}, "
            f"Health={self.health:.1f}, "
            f"Network={self.network:.1f}"
        )

In [6]:
# ==========================================================
# Cell 5 - Initialize the 810 Agents
# ==========================================================

class SuccessModel(Model):

    def __init__(self, seed=RANDOM_SEED):

        super().__init__(seed=seed)

        # Store agents in a simple list for now.
        self.agents_list = []

        # --------------------------------------------------
        # Create exactly 10 agents for every personality
        # profile
        # --------------------------------------------------

        for _, profile in personality_df.iterrows():

            personality_profile = profile.to_dict()

            for _ in range(AGENTS_PER_PERSONALITY):

                agent = Person(
                    model=self,
                    unique_id=None,
                    personality_profile=personality_profile
                )

                self.agents_list.append(agent)

        # --------------------------------------------------
        # Verify total population
        # --------------------------------------------------

        assert len(self.agents_list) == NUM_AGENTS

        # --------------------------------------------------
        # Count agents per personality profile
        # --------------------------------------------------

        self.profile_counts = {}

        for agent in self.agents_list:

            profile_id = agent.profile_id

            self.profile_counts[profile_id] = (
                self.profile_counts.get(profile_id, 0) + 1
            )

        # --------------------------------------------------
        # Verify personality distribution
        # --------------------------------------------------

        assert len(self.profile_counts) == NUM_PERSONALITY_TYPES

        assert all(
            count == AGENTS_PER_PERSONALITY
            for count in self.profile_counts.values()
        )


# ==========================================================
# Create the model
# ==========================================================

model = SuccessModel()


# ==========================================================
# Verification
# ==========================================================

print(f"Total agents created: {len(model.agents_list)}")

print(
    f"Personality profiles: {len(model.profile_counts)}"
)

print(
    "Agents per personality profile: "
    f"{AGENTS_PER_PERSONALITY}"
)

print("Population initialization successful.")

Total agents created: 810
Personality profiles: 81
Agents per personality profile: 10
Population initialization successful.
